# 02 - XGBoost (gradient-boosted trees, early stopping)

Gradient-boosted trees for cancel-by-arrival. The tree count is chosen by **early stopping** on a temporal validation tail - not a fixed 600 - which is faster and avoids over/under-fitting.

Evaluated on the SAME **decision-time walk-forward** as every model (00 §12 /
`src.walkforward`), directly comparable to the LogReg baseline (01) and the hazard
model (08). Uses the **tree feature family** (raw columns - trees are invariant to
monotone transforms and untroubled by collinearity, so no `_log` twins, no scaling),
a **cost-optimal** operating point (not F1), and **PR-AUC + Brier** as headline KPIs.
Boosting is often mis-calibrated, so the pipeline wraps the model in isotonic
calibration. Heavy lifting lives in `src.training` / `src.scoring`; this is a driver.

## 0 - Setup

In [1]:
from __future__ import annotations
import sys, time
from pathlib import Path
_t0 = time.perf_counter()
def _step(m): print(f"  [{time.perf_counter()-_t0:5.2f}s] {m}", flush=True)
_here = Path.cwd().resolve()
while not (_here / "pyproject.toml").exists():
    if _here == _here.parent: raise RuntimeError("project root not found")
    _here = _here.parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))

import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.io as pio
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.inspection import permutation_importance
from src import load_clean_reservations, color, figures_dir, tables_dir
from src import walkforward as WF
from src.features import load_feature_roster, family_feature_lists
import src.training as T, src.scoring as sc
pio.templates.default = "plotly_white"
MODEL = "xgboost"
BRAND = {n: color(n) for n in ["yellow","blue","green","orange","pink","purple","red"]}
FIG_DIR = figures_dir()/MODEL; FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR = tables_dir()/MODEL;  TBL_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42; pd.set_option("display.max_columns", 80)
_step("setup done (plotly).")

  [77.60s] setup done (plotly).


## 1 - Data + tree feature family

The **tree** family keeps the raw skewed columns (linear models would take the
`_log` twins) and tolerates collinearity. Leakage columns (company / nationality /
language / check-in) are already excluded by the roster (00 §11).

In [2]:
clean = load_clean_reservations()
roster = load_feature_roster()
NUM, CAT = family_feature_lists(roster, "tree")
print(f"tree-family features: {len(NUM)} numeric + {len(CAT)} categorical")
print("  numeric    :", NUM)
print("  categorical:", CAT)
print("  target     : status (1 = cancel at/before arrival)")

tree-family features: 14 numeric + 7 categorical
  numeric    : ['adults_n', 'arrival_dow', 'arrival_month', 'diff_gross_cancellation_fee', 'gross_per_night', 'has_children', 'has_corporate_code', 'has_group', 'has_promo', 'is_weekend_arrival', 'lead_time_days', 'log_gross_amount', 'los_nights', 'ratePlan_isSubjectToCityTax']
  categorical: ['cancellationFee_name', 'channelCode', 'guaranteeType', 'property_name', 'ratePlan_category', 'stay_bucket', 'unitGroup_name']
  target     : status (1 = cancel at/before arrival)


## 2 - Deployment fit (retune) + decision-time walk-forward

`src.training.retrain(mode="retune")` searches structural hp and picks the tree count by **early stopping** (n_estimators=2000, early_stopping_rounds=50 on a temporal val), fits the calibrated pipeline on
ALL resolved data, reports the honest decision-time walk-forward metrics, and persists
the joblib + card the app's scoring path loads.

In [3]:
_step("retune + deploy (heavy)...")
deploy = T.retrain(MODEL, mode="retune")
print("persisted ->", deploy["persisted"]["joblib"])
print("chosen hp:", deploy["hyperparams"])
print("trained on", deploy["n_train_deploy"], "resolved rows")
pf = pd.DataFrame(deploy["walk_forward"]["per_fold"]); display(pf.round(4))
print("walk-forward aggregate (mean +/- std):",
      {k: f"{v['mean']:.4f}+/-{v['std']:.4f}" for k, v in deploy["walk_forward"]["aggregate"].items()})
if len(pf):
    fig = go.Figure()
    fig.add_scatter(x=pf["origin"], y=pf["auc"], mode="lines+markers", name="AUC", line=dict(color=BRAND["blue"]))
    fig.add_scatter(x=pf["origin"], y=pf["ap"],  mode="lines+markers", name="AP (PR-AUC)", line=dict(color=BRAND["orange"]))
    fig.update_layout(title=f"{MODEL} decision-time walk-forward: per-fold AUC / AP",
                      xaxis_title="fold origin", yaxis_title="score")
    fig.show()

  [78.08s] retune + deploy (heavy)...
[retrain:xgboost] roster changed vs deployed model - added=['diff_gross_cancellation_fee_log', 'gross_per_night_log', 'lead_time_days_log', 'los_nights_log'] removed=[]
persisted -> /Users/ruby.grambauer/Documents/DEV/overbookinganalyses/Data/02_xgboost_model.joblib
chosen hp: {'max_depth': 4, 'learning_rate': 0.24465498962794868, 'subsample': 0.9509357413523924, 'colsample_bytree': 0.7031766510860622, 'min_child_weight': 10, 'reg_lambda': 1.1191282495204613, 'reg_alpha': 4.295475449362941, 'n_estimators': 148}
trained on 183008 resolved rows


,fold,origin,n_train,n_test,auc,ap,brier,cost,cost_threshold
0,0,2026-04-19,162864,2953,0.7184,0.2562,0.1281,31120.0,0.5834
1,1,2026-05-03,166058,3146,0.7437,0.2559,0.1204,32400.0,0.5518
2,2,2026-05-17,169447,3394,0.7774,0.3044,0.1073,30400.0,0.4924
3,3,2026-05-31,172756,3507,0.7628,0.3385,0.1179,35260.0,0.4830
4,4,2026-06-14,176288,3297,0.7573,0.2801,0.1077,29680.0,0.6604
5,5,2026-06-28,179906,1429,0.7920,0.1548,0.0785,7040.0,0.4034


walk-forward aggregate (mean +/- std): {'auc': '0.7586+/-0.0258', 'ap': '0.2650+/-0.0624', 'brier': '0.1100+/-0.0173', 'cost': '27650.0000+/-10285.6308'}


## 3 - Pooled out-of-time predictions + calibration

Pool the per-fold OOS predictions (frozen retuned hp) into ONE decision-aligned
sample; report pooled KPIs and the reliability curve - the probabilities feed the
overbooking decision, so calibration matters as much as ranking.

In [4]:
_step("pooled out-of-time predictions...")
oos = T.walk_forward_predict(MODEL, n_folds=12)
yv, pv = oos["y_true"].to_numpy(), oos["y_prob"].to_numpy()
print(f"pooled OOS: n={len(oos):,}  base-rate={yv.mean():.3f}  | "
      f"AUC={roc_auc_score(yv,pv):.4f}  AP={average_precision_score(yv,pv):.4f}  "
      f"Brier={brier_score_loss(yv,pv):.4f}")
from sklearn.calibration import calibration_curve
frac_pos, mean_pred = calibration_curve(yv, pv, n_bins=10, strategy="quantile")
fig = go.Figure()
fig.add_scatter(x=mean_pred, y=frac_pos, mode="lines+markers", name=MODEL, line=dict(color=BRAND["blue"]))
fig.add_scatter(x=[0, float(pv.max())], y=[0, float(pv.max())], mode="lines",
                name="perfect", line=dict(color="grey", dash="dash"))
fig.update_layout(title="Calibration (reliability) on pooled OOS",
                  xaxis_title="mean predicted probability", yaxis_title="observed frequency")
fig.show()

  [2360.54s] pooled out-of-time predictions...
pooled OOS: n=34,028  base-rate=0.125  | AUC=0.7338  AP=0.2563  Brier=0.1162


## 4 - Cost-optimal operating point (not F1)

Walking a guest costs far more than an empty room, so the threshold is
cost-minimising, tuned on the pooled OOS (`src.scoring`).

In [5]:
thr = sc.cost_threshold_from_scores(yv, pv)
op = sc.cost_at_threshold(yv, pv, thr)
print(f"cost-optimal threshold = {thr:.3f}  (walk={sc.COST_WALK:.0f} / empty={sc.COST_EMPTY:.0f})")
print(f"  precision={op['precision']:.3f}  recall={op['recall']:.3f}  total_cost={op['total_cost']:,.0f}")
grid = np.linspace(0.05, 0.95, 60)
costs = [sc.cost_at_threshold(yv, pv, t)["total_cost"] for t in grid]
fig = go.Figure(go.Scatter(x=grid, y=costs, mode="lines", line=dict(color=BRAND["orange"])))
fig.add_vline(x=thr, line=dict(color=BRAND["green"], dash="dash"), annotation_text=f"cost-opt {thr:.2f}")
fig.update_layout(title="Total cost vs decision threshold (minimum = operating point)",
                  xaxis_title="threshold", yaxis_title="total cost")
fig.show()

cost-optimal threshold = 0.645  (walk=300 / empty=80)
  precision=1.000  recall=0.002  total_cost=340,400


## 5 - Explainability (XAI)

Model-agnostic **permutation importance** (AP drop) on the deployed calibrated
pipeline, plus a partial-dependence curve for the strongest numeric driver.

In [6]:
model = sc.load_model(MODEL)
df = WF.add_outcome_known_date(clean)
X = df[NUM + CAT]; y = pd.to_numeric(df["status"], errors="coerce").astype(int)
samp = X.sample(n=min(5000, len(X)), random_state=RANDOM_STATE)
pi = permutation_importance(model, samp, y.loc[samp.index],
                            scoring="average_precision", n_repeats=5, random_state=RANDOM_STATE)
imp = (pd.DataFrame({"feature": NUM + CAT, "importance": pi.importances_mean})
         .sort_values("importance", ascending=False))
top = imp.head(15)
fig = go.Figure(go.Bar(x=top["importance"][::-1], y=top["feature"][::-1],
                       orientation="h", marker_color=BRAND["blue"]))
fig.update_layout(title=f"{MODEL} permutation importance (mean AP decrease)", xaxis_title="AP drop")
fig.show()

# Partial dependence for the strongest NUMERIC feature.
top_num = next((f for f in imp["feature"] if f in NUM), NUM[0])
qs = np.quantile(pd.to_numeric(samp[top_num], errors="coerce").dropna(), np.linspace(0.02, 0.98, 25))
base = samp.copy(); pdp = []
for v in qs:
    b = base.copy(); b[top_num] = v
    pdp.append(float(model.predict_proba(b)[:, 1].mean()))
fig = go.Figure(go.Scatter(x=qs, y=pdp, mode="lines+markers", line=dict(color=BRAND["orange"])))
fig.update_layout(title=f"Partial dependence: {top_num} -> P(cancel)",
                  xaxis_title=top_num, yaxis_title="mean predicted P(cancel)")
fig.show()

## 6 - Verdict

The calibrated xgboost is a horizon-blind per-booking tree model on the raw feature
family, tuned with early stopping (data-driven tree count, not a fixed guess),
evaluated honestly on the decision-time walk-forward and calibrated for the
cost-based overbooking decision. Compared head-to-head against LogReg (01) and the
hazard (08 §5) on the matched estimand; where it loses to the hazard near arrival is
the time-varying risk a per-booking model cannot represent.

## Rigorous scoring diagnostics — XGBoost vs. baseline (+ SHAP)
Standard battery on the pooled out-of-time predictions (`yv`, `pv`) computed above: mean-rate baseline, ROC/AUC, precision–recall/AP, confusion matrix, precision/recall/F1 vs. threshold, and SHAP. Reuses the OOS scores — nothing heavy is recomputed.


In [7]:
import src.diagnostics as diag
# Reuse pooled OOS predictions (yv, pv) and the cost-optimal threshold (thr) from above.
diag.print_report(yv, pv, model_name=MODEL, threshold=thr)
diag.roc_curve_fig(yv, pv, MODEL).show()
diag.pr_curve_fig(yv, pv, MODEL).show()
diag.confusion_fig(yv, pv, thr, MODEL).show()
diag.threshold_sweep_fig(yv, pv, thr).show()

=== Scoring diagnostics: xgboost  (n=34,028, threshold=0.645) ===
          metric  xgboost baseline
         ROC-AUC   0.7338      0.5
           PR-AP   0.2563   0.1253
           Brier   0.1162   0.1096
        log-loss   0.3909   0.3773
precision @ 0.64   1.0000         
   recall @ 0.64   0.0019         
       F1 @ 0.64   0.0037         
 accuracy @ 0.64   0.8750         
  base rate (p0)   0.1253   0.1253

Confusion @ 0.645:  TP=8  FP=0  FN=4255  TN=29765


In [8]:
# SHAP on the DEPLOYED XGBoost: transform features through the pipeline's preprocessing,
# then unwrap the calibrated pipeline to the underlying booster and explain it.
Xs = df[NUM + CAT].sample(n=min(2000, len(df)), random_state=RANDOM_STATE)
design, feat_names = diag.pipeline_design_matrix(model, Xs)
booster = diag.tree_estimator(model)
fig = diag.shap_bar_fig(booster, design, feat_names, title=f'{MODEL} SHAP (mean |value|)')
fig.show() if fig is not None else print('SHAP unavailable — see warning above')
diag.shap_beeswarm(booster, design, feat_names)   # native beeswarm (matplotlib)

SHAP unavailable — see warning above


/Users/ruby.grambauer/Documents/DEV/overbookinganalyses/src/diagnostics.py:286: UserWarning: shap_values: TreeExplainer failed (DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:cat__cancellationFee_name_Airbnb: object, cat__cancellationFee_name_Airbnb Longstay: object, cat__cancellationFee_name_Corporate Longstay: object, cat__cancellationFee_name_Corporate Mediumstay: object, cat__cancellationFee_name_Corporate Midstay: object, cat__cancellationFee_name_Corporate Shortstay: object, cat__cancellationFee_name_Flexible: object, cat__cancellationFee_name_Flexible Corporate Mediumstay: object, cat__cancellationFee_name_Flexible Corporate Midstay: object, cat__cancellationFee_name_Flexible Corporate Shortstay: object, cat__cancellationFee_name_Flexible Longstay: object, cat__cancellationFee_name_Flexible Mediumstay: object, cat__cancellationFee_nam